<a href="https://colab.research.google.com/github/adiatray1-ui/ADITYA_ATRAY-2026B03110264-/blob/main/Aditya_Atray_2026B03110264.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Let's start by importing the only two libraries this project is allowed to use.
import numpy as np      # NumPy: fast maths on lists/arrays of numbers (we use it for median, percentile, where...)
import pandas as pd     # Pandas: lets us work with data as a table, called a DataFrame

# These three lines only change how tables are PRINTED - they do not change any data.
pd.set_option("display.max_columns", 30)                     # show up to 30 columns instead of hiding some
pd.set_option("display.width", 200)                          # let wide tables stay on one line
pd.set_option("display.float_format", "{:.2f}".format)       # show decimal numbers with 2 digits, e.g. 69.79

# The three subject columns are used again and again, so I store their names in one list.
# Later I can write df[SUBJECTS] instead of typing all three names every time.
SUBJECTS = ["Python_Marks", "Maths_Marks", "Physics_Marks"]

## Step 1 – Load and inspect the dataset


In [ ]:
# ------------------------------------------------------------------
# HOW THE DATA GETS LOADED (read this part first!)
#   * In Google Colab  -> you choose ONE method with LOAD_METHOD below:
#         "upload" : a button appears and you pick the CSV from your computer (easiest)
#         "drive"  : the CSV is read from your Google Drive (stays saved for next time)
# ------------------------------------------------------------------
LOAD_METHOD = "upload"          # change this to "drive" if you want to use Google Drive instead
DRIVE_PATH = "/content/drive/MyDrive/02_student_performance_extended.csv"   # only used when LOAD_METHOD = "drive"

# First, let's find out whether this notebook is running inside Google Colab.
# The module 'google.colab' exists ONLY inside Colab, so on a normal computer the import fails
# and we simply record IN_COLAB = False.
try:
    from google.colab import files, drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB and LOAD_METHOD == "upload":
    # Colab + upload: show a "Choose Files" button and wait until the file is uploaded.
    print("Click 'Choose Files' below and select 02_student_performance_extended.csv")
    uploaded = files.upload()
    if len(uploaded) == 0:                     # nothing came back, e.g. you pressed Cancel
        raise FileNotFoundError("No file uploaded. Run this cell again and click 'Choose Files'.")
    file_name = list(uploaded.keys())[0]       # name of the file you just uploaded
    df_raw = pd.read_csv(file_name)            # read it into a table (Colab saves it under that name)
    print("Loaded:", file_name)

elif IN_COLAB and LOAD_METHOD == "drive":
    # Colab + Drive: connect to Google Drive (a pop-up asks for permission), then read the file from there.
    drive.mount("/content/drive")
    df_raw = pd.read_csv(DRIVE_PATH)
    print("Loaded from Drive:", DRIVE_PATH)

else:
    # Not in Colab: read from the project's 'data' folder, or from the same folder as this notebook.
    try:
        df_raw = pd.read_csv("../data/02_student_performance_extended.csv")
    except FileNotFoundError:
        df_raw = pd.read_csv("02_student_performance_extended.csv")
    print("Loaded from the local folder")

# A quick safety check: does the file we loaded really contain the columns we need?
# If not, I stop right here with a clear message instead of failing later with a confusing error.
needed_columns = ["Student_ID", "Department", "Attendance_Percent", "Study_Hours_Per_Day", "Python_Marks",
                  "Maths_Marks", "Physics_Marks", "Assignments_Submitted", "Internet_Access", "Total_Marks", "Result"]
missing_columns = [c for c in needed_columns if c not in df_raw.columns]
if len(missing_columns) > 0:
    raise ValueError(f"This file is missing columns: {missing_columns}. Did you upload the right CSV?")

# From now on I clean 'df'. 'df_raw' stays untouched so we can always compare with the original.
df = df_raw.copy()

# shape gives (number of rows, number of columns)
print("Rows, columns:", df.shape)
print()
# dtypes shows the data type of every column: int = whole number, float = decimal number, str = text
print("Column types:")
print(df.dtypes)

Click 'Choose Files' below and select 02_student_performance_extended.csv


Saving 02_student_performance_extended.csv to 02_student_performance_extended (1).csv
Loaded: 02_student_performance_extended (1).csv
Rows, columns: (1050, 11)

Column types:
Student_ID                object
Department                object
Attendance_Percent       float64
Study_Hours_Per_Day      float64
Python_Marks               int64
Maths_Marks                int64
Physics_Marks              int64
Assignments_Submitted      int64
Internet_Access           object
Total_Marks                int64
Result                    object
dtype: object


In [ ]:
# info() is a very useful summary: it lists every column with its type and how many values are NOT missing.
# If a column shows fewer values than the others, that column has missing data - keep an eye out for it!
df.info()
print()

# head() shows the first 5 rows, so we can see what the data actually looks like.
print("First 5 rows:")
print(df.head())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1050 entries, 0 to 1049
Data columns (total 11 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   Student_ID             1050 non-null   object 
 1   Department             1050 non-null   object 
 2   Attendance_Percent     1029 non-null   float64
 3   Study_Hours_Per_Day    1050 non-null   float64
 4   Python_Marks           1050 non-null   int64  
 5   Maths_Marks            1050 non-null   int64  
 6   Physics_Marks          1050 non-null   int64  
 7   Assignments_Submitted  1050 non-null   int64  
 8   Internet_Access        1050 non-null   object 
 9   Total_Marks            1050 non-null   int64  
 10  Result                 1050 non-null   object 
dtypes: float64(2), int64(5), object(4)
memory usage: 90.4+ KB

First 5 rows:
  Student_ID  Department  Attendance_Percent  Study_Hours_Per_Day  Python_Marks  Maths_Marks  Physics_Marks  Assignments_Submitted Internet

In [ ]:
# describe() gives quick statistics for every number column:
# count, mean (average), std (spread), min, 25% / 50% / 75% quartiles, and max.
print("Summary statistics of number columns:")
print(df.describe())
print()

# For text columns, value_counts() counts how many times each category appears.
# Here I use it to see how many students are in each department, how many have internet, and how many passed.
print(df["Department"].value_counts())
print()
print(df["Internet_Access"].value_counts())
print()
print(df["Result"].value_counts())

Summary statistics of number columns:
       Attendance_Percent  Study_Hours_Per_Day  Python_Marks  Maths_Marks  Physics_Marks  Assignments_Submitted  Total_Marks
count             1029.00              1050.00       1050.00      1050.00        1050.00                1050.00      1050.00
mean                78.27                 3.26         72.34        67.26          69.80                   4.97       209.40
std                 11.80                 1.17         13.37        15.08          14.81                   2.99        26.61
min                 41.20                 0.20         34.00        26.00          -7.00                   0.00       131.00
25%                 70.60                 2.40         63.00        56.00          60.00                   2.00       192.00
50%                 78.60                 3.20         73.00        67.00          70.00                   5.00       210.00
75%                 87.00                 4.10         82.00        77.00          79.0

## Step 2 – Missing values and duplicate `Student_ID`s


In [ ]:
# Let's count the missing values in every column.
# isna() turns each cell into True (missing) or False (present), and sum() adds up the Trues per column.
print("Missing values per column:")
print(df.isna().sum())

Missing values per column:
Student_ID                0
Department                0
Attendance_Percent       21
Study_Hours_Per_Day       0
Python_Marks              0
Maths_Marks               0
Physics_Marks             0
Assignments_Submitted     0
Internet_Access           0
Total_Marks               0
Result                    0
dtype: int64


In [ ]:
# Now let's find the duplicate IDs.
# keep=False means "mark EVERY row whose ID appears more than once" (both the original and its copy),
# so we can look at the pairs side by side.
is_dup = df.duplicated(subset="Student_ID", keep=False)
dup_rows = df[is_dup].sort_values("Student_ID")     # keep only those rows and sort them so each pair sits together

print("Rows that share an ID with another row:", len(dup_rows))

# df.duplicated() with no subset checks whole rows: it counts rows that are 100% identical copies.
print("Rows that are 100% identical copies   :", df.duplicated().sum())
print()

# Look closely at this table: for each pair, are the other columns the same (exact copy) or different (conflict)?
print(dup_rows)

Rows that share an ID with another row: 8
Rows that are 100% identical copies   : 2

    Student_ID  Department  Attendance_Percent  Study_Hours_Per_Day  Python_Marks  Maths_Marks  Physics_Marks  Assignments_Submitted Internet_Access  Total_Marks Result
183    STU0184  Electrical               64.10                 1.70            80           42             74                      7              No          196   Pass
690    STU0184  Electrical               64.10                 1.70            80           42             74                      7              No          196   Pass
247    STU0248       Civil               55.00                 5.50            83           55             73                      9              No          211   Pass
818    STU0248         ECE               83.90                 2.00            93           73             58                      1             Yes          224   Pass
275    STU0276         CSE               88.20                 5.10   

In [ ]:
rows_before = len(df)        # remember how many rows we had, so we can report how many were removed

# drop_duplicates keeps the FIRST row for each Student_ID and deletes the later ones.
df = df.drop_duplicates(subset="Student_ID", keep="first")
df = df.reset_index(drop=True)          # after deleting rows the numbering has gaps; this renumbers them 0,1,2,...

n_dup_removed = rows_before - len(df)   # I save this number because I need it in the final summary

print("Rows before:", rows_before, "| rows removed:", n_dup_removed, "| rows now:", len(df))
print("Is every Student_ID unique now?", df["Student_ID"].is_unique)     # should print True

Rows before: 1050 | rows removed: 4 | rows now: 1046
Is every Student_ID unique now? True


## Step 3 – Fill missing `Attendance_Percent`


In [ ]:
# Before I fix anything, let's look at the numbers that justify my choice of method.
print("Missing attendance values:", df["Attendance_Percent"].isna().sum())
print("Overall mean  :", df["Attendance_Percent"].mean())
print("Overall median:", df["Attendance_Percent"].median())
print()

# groupby("Department") splits the table into one group per department.
# agg([...]) then calculates several statistics for each group at once.
# Notice that the department medians are a little different from each other - that's why I fill by department.
print("Attendance by department:")
print(df.groupby("Department")["Attendance_Percent"].agg(["count", "mean", "median"]))

Missing attendance values: 21
Overall mean  : 78.25941463414634
Overall median: 78.6

Attendance by department:
            count  mean  median
Department                     
CSE           201 79.50   80.60
Civil         173 76.40   77.10
ECE           252 79.07   78.70
Electrical    179 77.33   78.60
Mechanical    220 78.42   79.05


In [ ]:
# Step A: before filling anything, remember WHICH rows are missing (True = missing).
# This column lets us (or a teacher) see later exactly which values were estimated.
df["Attendance_Imputed"] = df["Attendance_Percent"].isna()
n_att_imputed = df["Attendance_Imputed"].sum()     # save the count for the final summary

# Step B: for EVERY student, calculate the median attendance of THEIR department.
# transform("median") returns a column the same length as df, so each row gets its own department's median.
dept_median = df.groupby("Department")["Attendance_Percent"].transform("median")

# Step C: put that median into the empty cells only. fillna leaves the real values untouched.
df["Attendance_Percent"] = df["Attendance_Percent"].fillna(dept_median)

print("Values filled in:", n_att_imputed)
print("Missing values left:", df["Attendance_Percent"].isna().sum())     # should be 0
print()
print("The rows we filled in:")
print(df[df["Attendance_Imputed"]][["Student_ID", "Department", "Attendance_Percent"]])

Values filled in: 21
Missing values left: 0

The rows we filled in:
    Student_ID  Department  Attendance_Percent
161    STU0162         CSE               80.60
190    STU0191       Civil               77.10
230    STU0231         ECE               78.70
268    STU0269  Electrical               78.60
294    STU0295  Mechanical               79.05
344    STU0345         CSE               80.60
352    STU0353  Mechanical               79.05
357    STU0358  Electrical               78.60
369    STU0370         CSE               80.60
474    STU0475         ECE               78.70
516    STU0517  Electrical               78.60
523    STU0524         CSE               80.60
537    STU0538         ECE               78.70
542    STU0543  Mechanical               79.05
681    STU0682  Electrical               78.60
780    STU0781         ECE               78.70
819    STU0820         ECE               78.70
824    STU0825         CSE               80.60
916    STU0917         CSE             

## Step 4 – Check marks outside 0–100
A mark must be between 0 and 100. I find every impossible mark, **flag** the row (`Score_Out_Of_Range`) and fix it:
- **Above 100 → set to 100.** It is probably a typing slip for a very high score, so 100 keeps the student near the top without inventing a bigger number.
- **Below 0 → replace with the department median for that subject.** A negative mark has no meaning. Setting it to 0 would assume the worst and might wrongly label the student "at risk".

In [ ]:
# Let's first check the lowest and highest mark in each subject.
# If min is below 0 or max is above 100, we already know there is a problem.
print(df[SUBJECTS].agg(["min", "max"]))

# A mark is "bad" if it is below 0 OR above 100.
# This gives a True/False table with the same shape as the three subject columns.
bad_cells = (df[SUBJECTS] < 0) | (df[SUBJECTS] > 100)

# any(axis=1) looks across each ROW: True if at least one of the three subjects in that row is bad.
df["Score_Out_Of_Range"] = bad_cells.any(axis=1)
n_range_rows = df["Score_Out_Of_Range"].sum()     # save the count for the final summary

print()
print("Rows with an impossible mark:", n_range_rows)
print(df[df["Score_Out_Of_Range"]][["Student_ID", "Department"] + SUBJECTS + ["Total_Marks"]])

     Python_Marks  Maths_Marks  Physics_Marks
min            34           26             -7
max           105          112            100

Rows with an impossible mark: 3
    Student_ID Department  Python_Marks  Maths_Marks  Physics_Marks  Total_Marks
952    STU0953        CSE            81           63             -7          137
989    STU0990        ECE            71          112             67          250
991    STU0992        CSE           105           68             60          233


In [ ]:
# Before changing anything, I save the bad rows so I can show a BEFORE and AFTER comparison.
bad_ids = df.loc[df["Score_Out_Of_Range"], "Student_ID"]
before = df.loc[df["Score_Out_Of_Range"], ["Student_ID"] + SUBJECTS].copy()

for subject in SUBJECTS:                                   # repeat the same fix for Python, Maths and Physics
    df.loc[df[subject] > 100, subject] = 100               # 1) any mark above 100 becomes exactly 100
    df.loc[df[subject] < 0, subject] = np.nan              # 2) any negative mark is treated as "unknown" (missing)
    subject_median = df.groupby("Department")[subject].transform("median")
    df[subject] = df[subject].fillna(subject_median)       # 3) fill the unknown marks with the department median
    df[subject] = df[subject].astype(int)                  # 4) turn the numbers back into whole numbers

print("BEFORE:")
print(before)
print()
print("AFTER:")
print(df[df["Student_ID"].isin(bad_ids)][["Student_ID"] + SUBJECTS])
print()

# Final check: after the fix, every mark must be between 0 and 100.
print("Lowest/highest mark now:", df[SUBJECTS].min().min(), "/", df[SUBJECTS].max().max())

BEFORE:
    Student_ID  Python_Marks  Maths_Marks  Physics_Marks
952    STU0953            81           63             -7
989    STU0990            71          112             67
991    STU0992           105           68             60

AFTER:
    Student_ID  Python_Marks  Maths_Marks  Physics_Marks
952    STU0953            81           63             69
989    STU0990            71          100             67
991    STU0992           100           68             60

Lowest/highest mark now: 23 / 100


## Step 5 – Total and average marks
The old `Total_Marks` was added up **before** cleaning, so it is wrong for the rows I fixed. I recalculate it, then create `Average_Marks = Total ÷ 3`. Each subject is out of 100, so the average is also a **percentage**.

In [ ]:
df["Total_Marks_Recorded"] = df["Total_Marks"]            # keep the OLD total in a new column, so we can compare
df["Total_Marks"] = df[SUBJECTS].sum(axis=1)              # add the 3 subject columns together for each row (axis=1 = across the row)
df["Average_Marks"] = df["Total_Marks"] / 3               # average of the three subjects = a percentage

# Which totals changed after cleaning? Only the rows where I fixed an impossible mark should differ.
changed = df["Total_Marks"] != df["Total_Marks_Recorded"]
print("Rows where the total changed:", changed.sum())
print(df[changed][["Student_ID"] + SUBJECTS + ["Total_Marks_Recorded", "Total_Marks", "Average_Marks"]])
print()

# A quick overview of the new Average_Marks column.
print(df["Average_Marks"].describe())

Rows where the total changed: 3
    Student_ID  Python_Marks  Maths_Marks  Physics_Marks  Total_Marks_Recorded  Total_Marks  Average_Marks
952    STU0953            81           63             69                   137          213          71.00
989    STU0990            71          100             67                   250          238          79.33
991    STU0992           100           68             60                   233          228          76.00

count   1046.00
mean      69.79
std        8.83
min       43.67
25%       64.00
50%       70.00
75%       75.67
max       94.67
Name: Average_Marks, dtype: float64


## Step 6 – Grades with `pd.cut()`
`pd.cut()` sorts numbers into labelled ranges ("bins"). With `right=False`, each range **includes** its lower number and **excludes** the upper one.

| Grade | Average marks |
|---|---|
| F | below 40 |
| D | 40 to under 50 |
| C | 50 to under 60 |
| B | 60 to under 70 |
| A | 70 to under 80 |
| A+ | 80 and above |

In [ ]:
# These numbers are the EDGES of the ranges: 0-40, 40-50, 50-60, 60-70, 70-80, 80-100.
# I use 100.01 as the last edge so that a student with exactly 100 still falls inside the last range.
grade_edges = [0, 40, 50, 60, 70, 80, 100.01]
grade_names = ["F", "D", "C", "B", "A", "A+"]     # one label for each range, in the same order

# pd.cut puts each student's Average_Marks into the right range and returns that range's label.
df["Grade"] = pd.cut(df["Average_Marks"], bins=grade_edges, labels=grade_names, right=False)

# Count how many students got each grade. reindex(grade_names) keeps the grades in order F -> A+.
grade_counts = df["Grade"].value_counts().reindex(grade_names)
grade_percent = grade_counts / len(df) * 100       # turn the counts into percentages

print("Grade distribution:")
print(pd.DataFrame({"Students": grade_counts, "Percent": grade_percent}))
print()

# crosstab builds a "counting table": rows = departments, columns = grades.
print("Grades by department:")
print(pd.crosstab(df["Department"], df["Grade"]))

Grade distribution:
       Students  Percent
Grade                   
F             0     0.00
D            11     1.05
C           128    12.24
B           381    36.42
A           383    36.62
A+          143    13.67

Grades by department:
Grade       D   C    B   A  A+
Department                    
CSE         3  28   70  78  28
Civil       1  25   56  70  23
ECE         4  36  100  72  45
Electrical  1  18   68  81  15
Mechanical  2  21   87  82  32


## Step 7 – Department-wise summary


In [ ]:
# np.where(condition, value_if_true, value_if_false) is a "one-line if/else" for a whole column.
# Here: 100 if the student passed, otherwise 0. The average of this column is then the pass rate in %.
df["Pass_Pct"] = np.where(df["Result"] == "Pass", 100, 0)

# Named aggregation: NewColumnName=("column to use", "what to calculate")
dept_summary = df.groupby("Department").agg(
    Students=("Student_ID", "count"),         # how many students are in the department
    Mean_Avg=("Average_Marks", "mean"),       # the average of their average marks
    Median_Avg=("Average_Marks", "median"),   # the middle value (not affected by extreme students)
    Pass_Rate=("Pass_Pct", "mean"),           # percentage of students who passed
)
dept_summary = dept_summary.sort_values("Mean_Avg", ascending=False)   # best department at the top

print(dept_summary)
print()

# idxmax()/idxmin() return the NAME (index label) of the row with the biggest/smallest value.
print("Highest mean:", dept_summary["Mean_Avg"].idxmax(), "| Lowest mean:", dept_summary["Mean_Avg"].idxmin())

            Students  Mean_Avg  Median_Avg  Pass_Rate
Department                                           
Civil            175     70.46       70.33      97.71
Mechanical       224     70.29       70.17      98.66
Electrical       183     69.74       70.33      97.81
CSE              207     69.52       70.00      94.20
ECE              257     69.15       69.00      94.16

Highest mean: Civil | Lowest mean: ECE


## Step 8 – Top 10 students by `Average_Marks`


In [ ]:
# Sort all students by Average_Marks (highest first), then head(10) keeps only the first 10 rows.
top10 = df.sort_values("Average_Marks", ascending=False).head(10)

# Show only the columns that matter for a "leaderboard".
top10 = top10[["Student_ID", "Department", "Python_Marks", "Maths_Marks", "Physics_Marks", "Average_Marks", "Grade"]]

top10 = top10.reset_index(drop=True)     # renumber the rows 0..9
top10.index = top10.index + 1            # then shift to 1..10 so the row number works as the RANK
print(top10)

   Student_ID  Department  Python_Marks  Maths_Marks  Physics_Marks  Average_Marks Grade
1     STU0790  Mechanical            86          100             98          94.67    A+
2     STU0927         ECE            83          100             99          94.00    A+
3     STU0964         ECE            95           88             99          94.00    A+
4     STU0810         ECE           100           78            100          92.67    A+
5     STU0838       Civil            97           90             89          92.00    A+
6     STU0659  Mechanical            86           96             93          91.67    A+
7     STU0994       Civil            81          100             90          90.33    A+
8     STU0007  Mechanical           100           75             96          90.33    A+
9     STU0396       Civil            93           88             89          90.00    A+
10    STU0617         CSE           100           93             76          89.67    A+


## Step 9 – At-risk students (`Average_Marks < 40%`)


In [ ]:
# Label every student: "At Risk" if the average is below 40, otherwise "Not At Risk".
df["Risk_Status"] = np.where(df["Average_Marks"] < 40, "At Risk", "Not At Risk")

# Boolean filtering: keep only the rows where Risk_Status is "At Risk".
at_risk = df[df["Risk_Status"] == "At Risk"]
print("At-risk students (average < 40%):", len(at_risk), "out of", len(df))

# It is possible that NOBODY is below 40. In that case I don't want an empty, confusing output,
# so I print the 10 lowest averages instead (nsmallest = the n smallest values) to show who is closest to the line.
if len(at_risk) == 0:
    print("\nNobody is below 40%. The 10 lowest averages (closest to the line):")
    print(df.nsmallest(10, "Average_Marks")[["Student_ID", "Department"] + SUBJECTS + ["Average_Marks", "Result"]])
else:
    print(at_risk[["Student_ID", "Department"] + SUBJECTS + ["Average_Marks", "Result"]])

At-risk students (average < 40%): 0 out of 1046

Nobody is below 40%. The 10 lowest averages (closest to the line):
    Student_ID  Department  Python_Marks  Maths_Marks  Physics_Marks  Average_Marks Result
143    STU0144  Mechanical            34           38             59          43.67   Fail
394    STU0395         CSE            49           29             54          44.00   Fail
795    STU0796  Electrical            44           38             53          45.00   Fail
537    STU0538         ECE            48           67             25          46.67   Pass
973    STU0974         CSE            68           27             46          47.00   Pass
180    STU0181         CSE            61           31             54          48.67   Fail
627    STU0628         ECE            60           35             51          48.67   Pass
925    STU0926       Civil            45           46             55          48.67   Pass
433    STU0434  Mechanical            51           39            

In [ ]:
# ---- Extra check: students who fail individual subjects ----
# (df[SUBJECTS] < 40) is a True/False table; sum(axis=1) counts the Trues in each row = number of failed subjects.
df["Subjects_Below_40"] = (df[SUBJECTS] < 40).sum(axis=1)

# A student "needs support" if they have at least one subject below 40 OR the dataset says Fail.
# The symbol | means OR. I store it as 1 (yes) / 0 (no) so I can average it later.
needs_help = (df["Subjects_Below_40"] > 0) | (df["Result"] == "Fail")
df["Needs_Support"] = np.where(needs_help, 1, 0)
n_support = df["Needs_Support"].sum()

print("Students needing support:", n_support, f"({n_support / len(df) * 100:.1f}% of all students)")
print()
print("Number of students below 40 in each subject:")
print((df[SUBJECTS] < 40).sum())
print()

# The average of a column of 1s and 0s is the SHARE of 1s, so multiplying by 100 gives a percentage.
support_by_dept = df.groupby("Department")["Needs_Support"].mean() * 100
print("% of students needing support, by department:")
print(support_by_dept.sort_values(ascending=False))
print()

# Finally, a watch-list: the students who need support, with the worst cases first.
print("Watch-list (first 15, most failing subjects first):")
watch = df[df["Needs_Support"] == 1].sort_values(["Subjects_Below_40", "Average_Marks"], ascending=[False, True])
print(watch[["Student_ID", "Department", "Attendance_Percent", "Study_Hours_Per_Day"] + SUBJECTS + ["Average_Marks", "Result"]].head(15))

Students needing support: 64 (6.1% of all students)

Number of students below 40 in each subject:
Python_Marks      2
Maths_Marks      34
Physics_Marks    31
dtype: int64

% of students needing support, by department:
Department
CSE          10.14
ECE           9.34
Civil         4.00
Electrical    3.83
Mechanical    2.23
Name: Needs_Support, dtype: float64

Watch-list (first 15, most failing subjects first):
    Student_ID  Department  Attendance_Percent  Study_Hours_Per_Day  Python_Marks  Maths_Marks  Physics_Marks  Average_Marks Result
143    STU0144  Mechanical               51.80                 5.10            34           38             59          43.67   Fail
69     STU0070         ECE               78.20                 2.40            83           38             32          51.00   Fail
552    STU0553         ECE               55.40                 1.60            81           37             39          52.33   Fail
394    STU0395         CSE               66.10             

## Step 10 – Average marks by attendance band
I sort attendance into three bands with `pd.cut()`:
- **Low:** below 75%
- **Moderate:** 75% to 85%
- **High:** above 85%

(Attendance has one decimal, so the edge `74.99` means "just below 75".)

In [ ]:
# The edges 0, 74.99, 85, 100 create three ranges: Low (up to 74.99), Moderate (up to 85), High (above 85).
att_edges = [0, 74.99, 85, 100]
att_names = ["Low", "Moderate", "High"]

# include_lowest=True makes sure a student with exactly 0% attendance would still be counted.
df["Attendance_Band"] = pd.cut(df["Attendance_Percent"], bins=att_edges, labels=att_names, include_lowest=True)

# For each band: how many students, their average marks, and their pass rate.
att_table = df.groupby("Attendance_Band", observed=False).agg(
    Students=("Student_ID", "count"),
    Mean_Avg=("Average_Marks", "mean"),
    Median_Avg=("Average_Marks", "median"),
    Pass_Rate=("Pass_Pct", "mean"),
)
print(att_table)

# The correlation coefficient r is a number between -1 and +1:
#   near 0 = no straight-line link, near +1 = marks go UP when attendance goes up.
r_att = df["Attendance_Percent"].corr(df["Average_Marks"])

# .loc["High", "Mean_Avg"] means: take the row named "High" and the column named "Mean_Avg".
att_gap = att_table.loc["High", "Mean_Avg"] - att_table.loc["Low", "Mean_Avg"]
print(f"\nCorrelation between attendance and average marks: r = {r_att:.2f}")
print(f"Average marks of 'High' attendance minus 'Low' attendance: {att_gap:.2f} marks")

                 Students  Mean_Avg  Median_Avg  Pass_Rate
Attendance_Band                                           
Low                   386     67.41       67.67      94.30
Moderate              354     70.76       70.67      98.02
High                  306     71.68       72.00      97.06

Correlation between attendance and average marks: r = 0.23
Average marks of 'High' attendance minus 'Low' attendance: 4.27 marks


## Step 11 – Study hours vs average marks
I measure the link in two ways:
1. **Correlation coefficient (r):** a number from −1 to +1.

| r | Meaning |
|---|---|
| 0 to 0.1 | almost no link |
| 0.1 to 0.3 | weak link |
| 0.3 to 0.5 | moderate link |
| above 0.5 | strong link |

2. **Grouped comparison:** average marks for students who study few vs many hours.

> ⚠️ A correlation shows that two things *move together*. It does **not** prove that one *causes* the other.

In [ ]:
# First way to measure the link: correlation, calculated with Pandas.
r_study = df["Study_Hours_Per_Day"].corr(df["Average_Marks"])
print(f"Correlation (Pandas): r = {r_study:.3f}")

# I repeat the calculation with NumPy as a cross-check - both should give the same answer.
# np.corrcoef returns a small 2x2 table; the [0, 1] picks the number we want (the link between the two columns).
r_numpy = np.corrcoef(df["Study_Hours_Per_Day"], df["Average_Marks"])[0, 1]
print(f"Correlation (NumPy) : r = {r_numpy:.3f}")

# np.polyfit draws the best straight line through all the points: Average_Marks = intercept + slope * Study_Hours.
# The slope tells us how many marks go with ONE extra hour of study.
slope, intercept = np.polyfit(df["Study_Hours_Per_Day"], df["Average_Marks"], 1)
print(f"Best-fit line: Average_Marks = {intercept:.2f} + {slope:.2f} x Study_Hours")
print(f"-> each extra study hour goes with about {slope:.2f} more marks (a pattern, not proof of cause)")

Correlation (Pandas): r = 0.232
Correlation (NumPy) : r = 0.232
Best-fit line: Average_Marks = 64.08 + 1.75 x Study_Hours
-> each extra study hour goes with about 1.75 more marks (a pattern, not proof of cause)


In [ ]:
# Second way: group students by how long they study and compare the groups.
study_edges = [0, 2, 3, 4, 5, 24]                          # hours per day: under 2, 2-3, 3-4, 4-5, 5 or more
study_names = ["<2h", "2-3h", "3-4h", "4-5h", "5h+"]

# right=False means each group INCLUDES its lower edge (e.g. exactly 2 hours belongs to "2-3h").
df["Study_Band"] = pd.cut(df["Study_Hours_Per_Day"], bins=study_edges, labels=study_names, right=False)

study_table = df.groupby("Study_Band", observed=False).agg(
    Students=("Student_ID", "count"),
    Mean_Avg=("Average_Marks", "mean"),
    Pass_Rate=("Pass_Pct", "mean"),
)
print(study_table)

            Students  Mean_Avg  Pass_Rate
Study_Band                               
<2h              138     67.20      92.03
2-3h             287     68.19      97.21
3-4h             325     69.74      96.62
4-5h             212     72.18      97.17
5h+               84     73.71      97.62


## Step 12 – With vs without internet access


In [ ]:
# Group the students by Internet_Access ("Yes" / "No") and compare the two groups.
net_table = df.groupby("Internet_Access").agg(
    Students=("Student_ID", "count"),
    Mean_Avg=("Average_Marks", "mean"),
    Median_Avg=("Average_Marks", "median"),
    Pass_Rate=("Pass_Pct", "mean"),
)
print(net_table)

# .loc["Yes", "Mean_Avg"] = take the row labelled "Yes" and the column "Mean_Avg".
# A positive gap means students WITH internet score higher; a negative gap means lower.
mean_gap = net_table.loc["Yes", "Mean_Avg"] - net_table.loc["No", "Mean_Avg"]
pass_gap = net_table.loc["Yes", "Pass_Rate"] - net_table.loc["No", "Pass_Rate"]
print(f"\nAverage marks: with access minus without = {mean_gap:.2f}")
print(f"Pass rate    : with access minus without = {pass_gap:.2f} percentage points")

                 Students  Mean_Avg  Median_Avg  Pass_Rate
Internet_Access                                           
No                    119     69.18       70.00      91.60
Yes                   927     69.87       70.00      96.98

Average marks: with access minus without = 0.70
Pass rate    : with access minus without = 5.38 percentage points


## Step 13 – NumPy statistics for `Python_Marks`
First I turn the column into a NumPy array, then I use NumPy functions:
- `np.median` – the middle value
- `np.std` – standard deviation (how spread out the marks are)
- `np.percentile` – **Q1** = 25th, **Q2** = 50th (same as the median), **Q3** = 75th percentile

In [ ]:
# to_numpy() converts the Pandas column into a plain NumPy array, so the np.* functions can work on it.
python_marks = df["Python_Marks"].to_numpy()

median = np.median(python_marks)                      # the middle mark: half the students are below, half above

# Standard deviation tells us how far marks typically are from the average. Bigger = more spread out.
# ddof=1 gives the "sample" standard deviation, the usual choice when the data is a sample of students.
std = np.std(python_marks, ddof=1)

# Quartiles split the sorted marks into four equal parts.
# Q1 = 25% of students scored below this, Q2 = 50% (the median), Q3 = 75%.
q1, q2, q3 = np.percentile(python_marks, [25, 50, 75])

print("Median:", median)
print("Standard deviation:", round(std, 2))
print("Q1 (25th percentile):", q1)
print("Q2 (50th percentile):", q2, "<- same as the median")
print("Q3 (75th percentile):", q3)

Median: 72.5
Standard deviation: 13.36
Q1 (25th percentile): 63.0
Q2 (50th percentile): 72.5 <- same as the median
Q3 (75th percentile): 82.0


## Step 14 – Outliers with the IQR method
1. **IQR** = Q3 − Q1 (the spread of the middle 50% of students).
2. Lower fence = Q1 − 1.5 × IQR
3. Upper fence = Q3 + 1.5 × IQR
4. Any mark outside the fences is a **potential outlier**.

An outlier is something to *look at*, not automatically a mistake – a very weak or very strong student can be a real outlier.

In [ ]:
# q1 and q3 come from Step 13. The IQR is the distance between them.
iqr = q3 - q1
lower_fence = q1 - 1.5 * iqr        # marks below this line are unusually LOW
upper_fence = q3 + 1.5 * iqr        # marks above this line are unusually HIGH

print("IQR:", iqr)
print("Lower fence:", lower_fence)
print("Upper fence:", upper_fence)

# Keep the students whose Python mark is below the lower fence OR above the upper fence (| means OR).
outliers = df[(df["Python_Marks"] < lower_fence) | (df["Python_Marks"] > upper_fence)]
print("\nNumber of outliers:", len(outliers))
print(outliers[["Student_ID", "Department", "Python_Marks", "Maths_Marks", "Physics_Marks", "Average_Marks", "Result"]])

IQR: 19.0
Lower fence: 34.5
Upper fence: 110.5

Number of outliers: 2
    Student_ID  Department  Python_Marks  Maths_Marks  Physics_Marks  Average_Marks Result
54     STU0055       Civil            34           83             58          58.33   Fail
143    STU0144  Mechanical            34           38             59          43.67   Fail


## Step 15 – Summary of findings and five recommendations


In [ ]:
# First I collect a few facts that I want to mention in the summary.
best_dept = dept_summary["Mean_Avg"].idxmax()            # department with the highest mean
worst_dept = dept_summary["Mean_Avg"].idxmin()           # department with the lowest mean
most_common_grade = grade_counts.idxmax()                # the grade that the most students got

# Now I print the summary. Each {...} below is replaced by a value calculated in the earlier steps.
print("SUMMARY OF FINDINGS")
print("-------------------")
print(f"1. Cleaning: removed {n_dup_removed} duplicate IDs, filled {n_att_imputed} missing attendance values, "
      f"and fixed {n_range_rows} rows with impossible marks. {len(df)} unique students remain.")
print(f"2. Grades: the most common grade is {most_common_grade} "
      f"({grade_percent[most_common_grade]:.1f}% of students). Overall average = {df['Average_Marks'].mean():.2f}%.")
print(f"3. Departments: {best_dept} has the highest mean ({dept_summary.loc[best_dept, 'Mean_Avg']:.2f}%), "
      f"{worst_dept} the lowest ({dept_summary.loc[worst_dept, 'Mean_Avg']:.2f}%).")
print(f"4. At risk (average < 40%): {len(at_risk)} students. Needing support (any subject < 40 or Fail): {n_support} students.")
print(f"5. Attendance: r = {r_att:.2f}. 'High' attendance students average {att_gap:.2f} marks more than 'Low'.")
print(f"6. Study hours: r = {r_study:.2f}. About {slope:.2f} extra marks per extra study hour in the best-fit line.")
print(f"7. Internet: average marks differ by {mean_gap:.2f}; pass rate differs by {pass_gap:.2f} percentage points.")
print(f"8. Python_Marks: median = {median}, std = {std:.1f}, Q1 = {q1}, Q3 = {q3}, outliers (IQR rule) = {len(outliers)}.")

SUMMARY OF FINDINGS
-------------------
1. Cleaning: removed 4 duplicate IDs, filled 21 missing attendance values, and fixed 3 rows with impossible marks. 1046 unique students remain.
2. Grades: the most common grade is A (36.6% of students). Overall average = 69.79%.
3. Departments: Civil has the highest mean (70.46%), ECE the lowest (69.15%).
4. At risk (average < 40%): 0 students. Needing support (any subject < 40 or Fail): 64 students.
5. Attendance: r = 0.23. 'High' attendance students average 4.27 marks more than 'Low'.
6. Study hours: r = 0.23. About 1.75 extra marks per extra study hour in the best-fit line.
7. Internet: average marks differ by 0.70; pass rate differs by 5.38 percentage points.
8. Python_Marks: median = 72.5, std = 13.4, Q1 = 63.0, Q3 = 82.0, outliers (IQR rule) = 2.


In [ ]:
# For the recommendations I need a few more facts from the tables I built earlier.
low_att, high_att = att_table.loc["Low"], att_table.loc["High"]       # the "Low" and "High" rows of the attendance table
low_study, high_study = study_table.iloc[0], study_table.iloc[-1]     # first and last rows of the study-hours table

subject_fails = (df[SUBJECTS] < 40).sum()                             # how many students are below 40 in each subject
hardest_subject = subject_fails.idxmax().replace("_Marks", "")        # the subject with the most low marks (name without "_Marks")

# Students whose AVERAGE is fine (40 or more) but who still failed at least one subject.
# An "average only" rule would miss them - that's why I screen by subject too.
hidden = df[(df["Subjects_Below_40"] > 0) & (df["Average_Marks"] >= 40)]
top_support_dept = support_by_dept.idxmax()                           # department with the highest % of students needing support

# Recommendation 5 changes its wording depending on how big the internet pass-rate gap is.
# If the gap is 3 points or more I suggest extra support; otherwise I say internet is probably not the main barrier.
if abs(pass_gap) >= 3:
    internet_text = (f"Students without internet access have a pass rate {abs(pass_gap):.1f} points {'lower' if pass_gap > 0 else 'higher'}, "
                     f"although their average marks are similar (gap {abs(mean_gap):.2f}). Offer extra lab/library hours and offline study packs, "
                     f"and check which subjects these {int(net_table.loc['No', 'Students'])} students struggle with.")
else:
    internet_text = (f"Internet access shows only a small difference (marks gap {abs(mean_gap):.2f}, pass-rate gap {abs(pass_gap):.1f} points), "
                     f"so it is probably not the main barrier. Focus first on attendance, study skills and subject support.")

print("FIVE RECOMMENDATIONS FOR ACADEMIC SUPPORT")
print("-----------------------------------------")
print(f"1. Attendance alerts: 'Low' attendance students average {low_att['Mean_Avg']:.1f} marks vs {high_att['Mean_Avg']:.1f} for 'High' "
      f"(r = {r_att:.2f}). Contact students when attendance falls below 75% ({int(low_att['Students'])} students are below it now).")
print(f"2. Study coaching: average marks rise from {low_study['Mean_Avg']:.1f} ({low_study.name}) to {high_study['Mean_Avg']:.1f} ({high_study.name}) "
      f"as study time increases (r = {r_study:.2f}). Run time-management workshops for the {int(low_study['Students'])} students studying under 2 hours a day.")
print(f"3. Check subjects, not just averages: {len(hidden)} students have an average of 40% or more but scored below 40 in a subject. "
      f"{hardest_subject} has the most low marks ({int(subject_fails.max())}), so start remedial classes there.")
print(f"4. Department mentoring: {top_support_dept} has the highest share of students needing support ({support_by_dept.max():.1f}% "
      f"vs {n_support / len(df) * 100:.1f}% overall). Start mentoring and bridge sessions there.")
print(f"5. Internet access: {internet_text}")

# Finally, I always add the limitations, so nobody reads more into these patterns than the data can support.
print()
print("IMPORTANT CAVEATS")
print("-----------------")
print("- Correlation is NOT causation: motivation or earlier preparation could affect both study time and marks.")
print("- Part of this dataset (STU0501 onward) is synthetic, so results show the METHOD, not a real cohort.")
print("- Only a handful of rows were fixed (imputed attendance, corrected marks); they are flagged in the data.")

FIVE RECOMMENDATIONS FOR ACADEMIC SUPPORT
-----------------------------------------
1. Attendance alerts: 'Low' attendance students average 67.4 marks vs 71.7 for 'High' (r = 0.23). Contact students when attendance falls below 75% (386 students are below it now).
2. Study coaching: average marks rise from 67.2 (<2h) to 73.7 (5h+) as study time increases (r = 0.23). Run time-management workshops for the 138 students studying under 2 hours a day.
3. Check subjects, not just averages: 64 students have an average of 40% or more but scored below 40 in a subject. Maths has the most low marks (34), so start remedial classes there.
4. Department mentoring: CSE has the highest share of students needing support (10.1% vs 6.1% overall). Start mentoring and bridge sessions there.
5. Internet access: Students without internet access have a pass rate 5.4 points lower, although their average marks are similar (gap 0.70). Offer extra lab/library hours and offline study packs, and check which subjects 